# Stablecoin Peg Stability & Risk Analysis

**Research Question:**  
How tightly do major stablecoins maintain their $1 peg, under what conditions do they deviate, and what does this imply for risk-aware capital allocation in DeFi?

| Stablecoin | Ticker | Mechanism |
|---|---|---|
| Tether | USDT | Fiat-collateralised (centralised) |
| USD Coin | USDC | Fiat-collateralised (centralised) |
| Dai | DAI | Crypto-collateralised (MakerDAO) |
| Frax | FRAX | Fractional-algorithmic (hybrid) |

**Data source:** [CoinGecko Public API](https://www.coingecko.com/en/api) — daily close prices + volumes, last 365 days  
**Author:** Engin Samet Dede

In [ ]:
import time
import copy
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path

CHART_DIR = Path("charts")
CHART_DIR.mkdir(exist_ok=True)

PEG        = 1.0
TIGHT_BAND = 0.005
DEPEG_BAND = 0.01

COINS = {
    "USDT": "tether",
    "USDC": "usd-coin",
    "DAI" : "dai",
    "FRAX": "frax",
}
COLORS = {
    "USDT": "#26A17B",
    "USDC": "#2775CA",
    "DAI" : "#F4B731",
    "FRAX": "#000000",
}

def save_chart(fig, filename, width=1000, height=450):
    """Write image — converts Timestamp x-values to str for kaleido 1.x compatibility."""
    fig2 = copy.deepcopy(fig)
    for trace in fig2.data:
        if hasattr(trace, 'x') and trace.x is not None:
            trace.x = [
                x.strftime('%Y-%m-%d') if hasattr(x, 'strftime') else x
                for x in trace.x
            ]
    fig2.write_image(str(CHART_DIR / filename), width=width, height=height, scale=2)


## 1. Data Collection

We pull 365 days of daily USD prices **and trading volumes** from the CoinGecko `/coins/{id}/market_chart` endpoint. Volumes are needed for the volume-weighted deviation analysis in Section 5.6. BTC and ETH prices are fetched separately as market-stress proxies for Section 5.7.

In [ ]:
def fetch_market_data(coin_id: str, days: int = 365, retries: int = 5):
    """Returns (price_series, volume_series) indexed by date."""
    url    = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days, "interval": "daily"}

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=45)
            if r.status_code == 429 and attempt < retries:
                ra = r.headers.get("Retry-After")
                wait = int(ra) if ra and ra.isdigit() else 15 * attempt
                print(f"  rate-limited; retrying in {wait}s", flush=True)
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()

            def to_series(raw, name):
                df = pd.DataFrame(raw, columns=["ts", name])
                df["date"] = pd.to_datetime(df["ts"], unit="ms").dt.normalize()
                return df.set_index("date")[name]

            return to_series(data["prices"], "price"), to_series(data["total_volumes"], "volume")
        except requests.RequestException as exc:
            if attempt == retries:
                raise
            wait = 5 * attempt
            print(f"  attempt {attempt}/{retries} failed ({exc.__class__.__name__}); retrying in {wait}s", flush=True)
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {coin_id} after {retries} attempts.")


# --- stablecoins ---
raw_prices, raw_volumes = {}, {}
for ticker, coin_id in COINS.items():
    print(f"Fetching {ticker} ...", end=" ", flush=True)
    p, v = fetch_market_data(coin_id)
    raw_prices[ticker] = p
    raw_volumes[ticker] = v
    print(f"{len(p)} rows")
    time.sleep(8)

prices  = pd.concat(raw_prices,  axis=1)
volumes = pd.concat(raw_volumes, axis=1)
prices.index.name = volumes.index.name = "date"

# --- BTC & ETH for market-stress correlation ---
print("Fetching BTC ...", end=" ", flush=True)
btc_price, _ = fetch_market_data("bitcoin")
print(f"{len(btc_price)} rows");  time.sleep(8)

print("Fetching ETH ...", end=" ", flush=True)
eth_price, _ = fetch_market_data("ethereum")
print(f"{len(eth_price)} rows")

print(f"\nDataset : {prices.shape[0]} days x {prices.shape[1]} stablecoins")
print(f"Period  : {prices.index.min().date()} -> {prices.index.max().date()}")
prices.tail(3)

## 2. Data Cleaning

Trim to the last 365 complete days and forward-fill at most 1–2 isolated missing values (exchange non-reporting days). Verify no structural gaps remain.

In [ ]:
def dedup(df):
    return df[~df.index.duplicated(keep='last')].sort_index()

prices  = dedup(prices).iloc[-365:].copy()
volumes = dedup(volumes).reindex(prices.index).ffill()
prices  = prices.ffill()

print('Remaining NaNs after forward-fill:')
print(prices.isnull().sum())
print('\nBasic statistics:')
prices.describe().round(6)


## 3. Peg Deviation Analysis

For each stablecoin on each day we compute the **absolute deviation** from $1.00:

$$\delta_t = |p_t - 1|$$

Risk metrics:
- **Mean / median deviation** — average distance from peg
- **Max deviation** — worst single-day move
- **Volatility (σ)** — standard deviation of daily deviation
- **Days outside ±0.5 % / ±1.0 %** — frequency of stress events

In [ ]:
dev = (prices - PEG).abs()

summary = pd.DataFrame({
    "Mean deviation ($)"   : dev.mean(),
    "Median deviation ($)" : dev.median(),
    "Max deviation ($)"    : dev.max(),
    "Volatility (σ)"       : dev.std(),
    f"Days outside ±{TIGHT_BAND*100:.0f}%": (dev > TIGHT_BAND).sum(),
    f"Days outside ±{DEPEG_BAND*100:.0f}%": (dev > DEPEG_BAND).sum(),
    f"% inside ±{TIGHT_BAND*100:.0f}% band": (dev <= TIGHT_BAND).mean() * 100,
}).T
summary.columns.name = "Stablecoin"
print(summary.round(6).to_string())

## 4. De-peg Event Detection

A **de-peg event** is any day where `|price − 1| > 1%`. We isolate these days and classify them as premium (price > $1) or discount (price < $1).

In [ ]:
signed_dev = prices - PEG

for ticker in COINS:
    events = signed_dev[ticker][dev[ticker] > DEPEG_BAND].sort_index()
    if events.empty:
        print(f"{ticker}: no de-peg events (all days within ±1%)")
    else:
        direction = events.apply(lambda x: "premium" if x > 0 else "discount")
        print(f"\n{ticker} — {len(events)} de-peg event(s):")
        print(pd.DataFrame({
            "price"         : prices[ticker][events.index],
            "deviation ($)" : events.round(5),
            "direction"     : direction,
        }).to_string())

## 4.5 De-peg Recovery Time Analysis

For each de-peg event we measure how many consecutive days the stablecoin remained outside the ±1% band before returning. Recovery time is a critical portfolio risk metric: a 3-day recovery (USDC/SVB 2023) is fundamentally different from a permanent de-peg (UST 2022) even if peak deviation looks similar.

In [ ]:
def depeg_recovery(ticker: str) -> pd.DataFrame:
    s = dev[ticker]
    records, i = [], 0
    dates, vals = s.index.tolist(), s.values.tolist()
    while i < len(vals):
        if vals[i] > DEPEG_BAND:
            start   = dates[i]
            max_dev = vals[i]
            j = i + 1
            while j < len(vals) and vals[j] > DEPEG_BAND:
                max_dev = max(max_dev, vals[j])
                j += 1
            end       = dates[j] if j < len(vals) else None
            duration  = (dates[j] - start).days if end else None
            direction = "discount" if prices[ticker].loc[start] < PEG else "premium"
            records.append({
                "start"            : start.date(),
                "end"              : end.date() if end else "ongoing",
                "duration (days)"  : duration if duration is not None else ">365",
                "max deviation ($)": round(max_dev, 5),
                "direction"        : direction,
            })
            i = j
        else:
            i += 1
    return pd.DataFrame(records)


print("=== De-peg Recovery Time Analysis ===\n")
for ticker in COINS:
    df = depeg_recovery(ticker)
    if df.empty:
        print(f"{ticker}: no de-peg events in period.\n")
    else:
        num_dur = pd.to_numeric(df["duration (days)"], errors="coerce")
        print(f"{ticker}: {len(df)} event(s), avg recovery {num_dur.mean():.1f} days")
        print(df.to_string(index=False))
        print()

## 5. Visualisations

### 5.1 — Daily Price Time Series with Peg Bands

In [ ]:
fig = go.Figure()
x_band = list(prices.index) + list(prices.index[::-1])
fig.add_trace(go.Scatter(x=x_band,
    y=[1+TIGHT_BAND]*len(prices) + [1-TIGHT_BAND]*len(prices),
    fill="toself", fillcolor="rgba(144,238,144,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="±0.5% band"))
fig.add_trace(go.Scatter(x=x_band,
    y=[1+DEPEG_BAND]*len(prices) + [1-DEPEG_BAND]*len(prices),
    fill="toself", fillcolor="rgba(255,165,0,0.10)",
    line=dict(color="rgba(0,0,0,0)"), name="±1.0% band"))
fig.add_hline(y=1.0, line_dash="dash", line_color="gray",
    annotation_text="$1.00 peg", annotation_position="top left")
for ticker in COINS:
    fig.add_trace(go.Scatter(x=prices.index, y=prices[ticker],
        mode="lines", name=ticker, line=dict(color=COLORS[ticker], width=1.5)))
fig.update_layout(title="Stablecoin Daily Price — 365 Days",
    xaxis_title="Date", yaxis_title="Price (USD)",
    hovermode="x unified", template="plotly_white",
    legend=dict(orientation="h", y=-0.15), height=500)
save_chart(fig, "price_timeseries.png", height=500)
fig.show()

### 5.2 — Deviation Distribution per Stablecoin (Box Plot)

In [ ]:
fig2 = go.Figure()
for ticker in COINS:
    fig2.add_trace(go.Box(y=dev[ticker], name=ticker,
        marker_color=COLORS[ticker], boxmean=True,
        jitter=0.3, pointpos=-1.5, boxpoints="outliers"))
fig2.add_hline(y=TIGHT_BAND, line_dash="dot", line_color="green",
    annotation_text="±0.5% threshold", annotation_position="top right")
fig2.add_hline(y=DEPEG_BAND, line_dash="dot", line_color="orange",
    annotation_text="±1.0% de-peg threshold", annotation_position="top right")
fig2.update_layout(title="Absolute Peg Deviation Distribution by Stablecoin",
    yaxis_title="|price - $1| (USD)",
    template="plotly_white", height=480, showlegend=False)
save_chart(fig2, "deviation_boxplot.png", height=480)
fig2.show()

### 5.3 — Monthly Average Deviation Heatmap

In [ ]:
dev_m = dev.copy()
dev_m["month"] = dev_m.index.to_period("M").astype(str)
hm = dev_m.groupby("month")[list(COINS.keys())].mean()
fig3 = px.imshow(hm.T * 100, color_continuous_scale="YlOrRd",
    labels=dict(x="Month", y="Stablecoin", color="Avg deviation (%)"),
    title="Monthly Average Peg Deviation (%) — Heatmap",
    aspect="auto", text_auto=".3f")
fig3.update_layout(template="plotly_white", height=320)
save_chart(fig3, "monthly_heatmap.png", height=320)
fig3.show()

### 5.4 — Risk Score Radar Chart

In [ ]:
radar_metrics = {
    "Mean deviation"     : dev.mean(),
    "Max deviation"      : dev.max(),
    "Volatility"         : dev.std(),
    "Days outside ±0.5%" : (dev > TIGHT_BAND).sum(),
    "Days outside ±1.0%" : (dev > DEPEG_BAND).sum(),
}
radar_df   = pd.DataFrame(radar_metrics)
radar_norm = radar_df.div(radar_df.max())
categories = list(radar_metrics.keys())

fig4 = go.Figure()
for ticker in COINS:
    vals = radar_norm.loc[ticker].tolist()
    fig4.add_trace(go.Scatterpolar(r=vals + [vals[0]],
        theta=categories + [categories[0]], fill="toself", name=ticker,
        line_color=COLORS[ticker], opacity=0.6))
fig4.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Relative Peg Risk Profile (normalised, higher = riskier)",
    template="plotly_white", height=500,
    legend=dict(orientation="h", y=-0.1))
save_chart(fig4, "risk_radar.png", height=500)
fig4.show()

### 5.5 — 30-Day Rolling Peg Volatility

Static mean deviation masks *when* risk is rising or falling. Rolling volatility (σ of `|price − $1|` over the trailing 30 days) shows the time-varying risk profile — essential for a portfolio that reacts to changing conditions rather than relying on backward-looking averages.

In [ ]:
rolling_vol = dev.rolling(30).std()

fig5 = go.Figure()
for ticker in COINS:
    fig5.add_trace(go.Scatter(x=rolling_vol.index, y=rolling_vol[ticker],
        mode="lines", name=ticker, line=dict(color=COLORS[ticker], width=1.5)))
fig5.update_layout(title="30-Day Rolling Peg Volatility",
    xaxis_title="Date", yaxis_title="Rolling sigma of |price - $1|",
    template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", y=-0.15), height=450)
save_chart(fig5, "rolling_volatility.png")
fig5.show()

### 5.6 — Volume-Weighted Peg Deviation

Simple deviation treats a $0.005 move on a \$50M volume day the same as on a \$5B day. Volume-weighting corrects this: deviations during high-activity periods carry more weight, producing a liquidity-adjusted risk signal more relevant to actual capital deployment.

In [ ]:
vw_rolling = {}
for ticker in COINS:
    vol = volumes[ticker].reindex(dev.index).ffill()
    vw_rolling[ticker] = (dev[ticker] * vol).rolling(30).sum() / vol.rolling(30).sum()
vw_df = pd.DataFrame(vw_rolling)

fig6 = go.Figure()
for ticker in COINS:
    fig6.add_trace(go.Scatter(x=vw_df.index, y=vw_df[ticker],
        mode="lines", name=ticker, line=dict(color=COLORS[ticker], width=1.5)))
fig6.update_layout(title="Volume-Weighted Peg Deviation (30-day rolling)",
    xaxis_title="Date", yaxis_title="VW deviation (USD)",
    template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", y=-0.15), height=450)
save_chart(fig6, "vw_deviation.png")
fig6.show()

comparison = pd.DataFrame({
    "Simple mean ($)": dev.mean().round(6),
    "VW mean ($)"    : vw_df.mean().round(6),
    "VW / simple"    : (vw_df.mean() / dev.mean()).round(3),
})
print("\nVolume-weighted vs simple mean deviation:")
print(comparison.to_string())

### 5.7 — BTC Market Stress vs Stablecoin Peg Deviation

We use `|BTC daily return|` as a market stress proxy and test its Pearson correlation with each stablecoin's deviation. If de-pegs cluster around broad market sell-offs, a risk model should condition stablecoin exposure on macro volatility — not treat it as idiosyncratic.

In [ ]:
btc        = btc_price.reindex(dev.index).ffill()
eth        = eth_price.reindex(dev.index).ffill()
btc_stress = btc.pct_change().abs().fillna(0)

print("Pearson correlation: |BTC 1-day return| vs stablecoin peg deviation\n")
rows = []
for ticker in COINS:
    aligned = pd.concat([btc_stress, dev[ticker]], axis=1).dropna()
    r, p = stats.pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    rows.append({"Stablecoin": ticker, "r": round(r, 4),
                 "p-value": round(p, 6), "Significant (p<0.05)": "Yes" if p < 0.05 else "No"})
print(pd.DataFrame(rows).set_index("Stablecoin").to_string())

fig7 = make_subplots(specs=[[{"secondary_y": True}]])
btc_norm = btc / btc.max()
fig7.add_trace(go.Scatter(x=btc.index, y=btc_norm, name="BTC (normalised)",
    line=dict(color="rgba(247,147,26,0.5)", width=1, dash="dot")), secondary_y=False)
for ticker in COINS:
    fig7.add_trace(go.Scatter(x=dev.index, y=dev[ticker], name=ticker,
        line=dict(color=COLORS[ticker], width=1.5)), secondary_y=True)
fig7.update_layout(title="BTC Price vs Stablecoin Peg Deviation",
    template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", y=-0.15), height=480)
fig7.update_yaxes(title_text="BTC (normalised)", secondary_y=False)
fig7.update_yaxes(title_text="|price - $1| (USD)", secondary_y=True)
save_chart(fig7, "btc_correlation.png", height=480)
fig7.show()

## 6. Statistical Tests

**Kruskal–Wallis** test checks whether deviation distributions differ significantly across stablecoins (non-parametric, appropriate given heavy tails). Post-hoc **Mann–Whitney U** identifies which pairs are statistically distinct.

In [ ]:
groups = [dev[t].dropna().values for t in COINS]
h_stat, p_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis  H = {h_stat:.2f},  p = {p_kw:.4e}")
if p_kw < 0.05:
    print("-> Distributions differ significantly (p < 0.05) — pairwise tests below.\n")

tickers = list(COINS.keys())
results = []
for i in range(len(tickers)):
    for j in range(i + 1, len(tickers)):
        a, b = tickers[i], tickers[j]
        u, p = stats.mannwhitneyu(dev[a].dropna(), dev[b].dropna(), alternative="two-sided")
        results.append({"Pair": f"{a} vs {b}", "U": round(u, 0),
                         "p-value": round(p, 6), "Significant": "Yes" if p < 0.05 else "No"})
pd.DataFrame(results).set_index("Pair")

## 7. Key Findings & Risk Assessment

### Peg Stability Ranking

| Rank | Stablecoin | Mechanism | Risk Profile |
|------|------------|-----------|-------------|
| 1st | USDT | Fiat-collateralised | Tightest day-to-day peg; opaque reserve composition creates discontinuous tail risk |
| 2nd | USDC | Fiat-collateralised | Highly stable; notable SVB de-peg (March 2023) resolved in ~3 days — short recovery |
| 3rd | DAI | Crypto-collateralised | Slightly wider baseline; fully transparent on-chain collateral makes risk observable |
| 4th | FRAX | Fractional-algorithmic | Widest distribution; reflexive mechanism amplifies de-peg risk during market stress |

### Capital Allocation Implications

1. **Fiat-backed (USDT, USDC)** show the tightest peg on average but carry **concentrated, discontinuous risk** — a single custodian failure triggers rapid de-pegs that historical σ cannot predict. Recovery time analysis shows these events resolve quickly (days), but the initial shock can be severe.

2. **DAI**'s higher baseline noise is offset by **full on-chain transparency** — collateral ratios are verifiable in real time, making it arguably the most monitorable risk in the set.

3. **FRAX**'s fractional mechanism creates **reflexive risk**: FXS collateral weakens precisely during market sell-offs, widening the peg at the worst moment. The BTC correlation analysis quantifies how much of this is driven by broad market stress.

4. **Volume-weighted deviation** shows that some apparent de-pegs occur on thin-volume days and carry less systemic weight than raw deviation numbers suggest.

5. **Rolling volatility** shows that peg risk is not stationary — stress clusters in specific periods, meaning a fixed risk budget for stablecoin exposure will be systematically mispriced.

6. The **Kruskal–Wallis test** confirms all four risk profiles are statistically distinct — grouping them as generic "$1 assets" in a portfolio model is not defensible.

### Conclusion

> A DeFi yielding book that treats all stablecoins as equivalent misprices risk along multiple dimensions simultaneously: mechanism type, recovery speed, volume-adjusted impact, and correlation with macro stress. A robust allocation framework must disaggregate stablecoin exposure by mechanism and monitor deviation metrics dynamically.